# hnc on Colab

Run the full Mapillary -> TRIBE v2 -> GeoParquet 2.0 pipeline on a Colab Pro **L4 24GB** runtime.

**Before you start**

1. **Runtime > Change runtime type > L4 GPU**.
2. Open the **Secrets** tab (key icon, left sidebar) and add:
   - `MAPILLARY_ACCESS_TOKEN` (from https://www.mapillary.com/dashboard/developers)
   - `HF_TOKEN` (from https://huggingface.co/settings/tokens, with read access, and accept the [Llama 3.2 license](https://huggingface.co/meta-llama/Llama-3.2-3B) on Hugging Face first)
3. Run all cells top to bottom.

Repo, https://github.com/walkthru-earth/hnc

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf hnc
!git clone --depth 1 https://github.com/walkthru-earth/hnc.git
%cd /content/hnc

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!uv --version

In [ ]:
from google.colab import userdata
import os

os.environ['MAPILLARY_ACCESS_TOKEN'] = userdata.get('MAPILLARY_ACCESS_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

with open('.env', 'w') as f:
    f.write(f"MAPILLARY_ACCESS_TOKEN={os.environ['MAPILLARY_ACCESS_TOKEN']}\n")
    f.write(f"HF_TOKEN={os.environ['HF_TOKEN']}\n")
print('secrets loaded, .env written')

In [ ]:
!uv sync --extra tribe 2>&1 | tail -20

In [ ]:
!uv run python -c "import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
!uv run hnc-run \
  --bbox-name camden \
  --max-images 50 \
  --device cuda \
  --out /content/hnc_camden.parquet

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
df = con.execute("SELECT image_id, captured_at, lon, lat, compass_angle, len(brain_activity) AS n_vertices FROM read_parquet('/content/hnc_camden.parquet') LIMIT 10").fetchdf()
df

In [ ]:
from google.colab import files
files.download('/content/hnc_camden.parquet')